In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Change data capture") \
    .master("local[*]") \
    .getOrCreate()

print("Spark created!")

In [ ]:
spark.sql("""
CREATE TABLE course.test.users (
    id bigint,
    name string,
    email string,
    city string
) 
USING iceberg
""")

In [ ]:
spark.sql("""
INSERT INTO course.test.users VALUES 
(1, 'Alice', 'alice@mail.com', 'Moscow'),
(2, 'Bob', 'bob@mail.com', 'Berlin')
""")

In [ ]:
spark.sql("""
INSERT INTO course.test.users VALUES 
(3, 'Carrol', 'carrol@mail.com', 'London')
""")

In [ ]:
spark.sql("""
UPDATE course.test.users SET city = 'Dubai' WHERE id = 1
""")

In [ ]:
spark.sql("""
DELETE FROM course.test.users WHERE id = 2
""")

In [ ]:
spark.sql("""
SELECT * FROM course.test.users
""").show()

In [ ]:
spark.sql("""
SELECT snapshot_id, parent_id
FROM course.test.users.history
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT 
    COALESCE(curr.id, prev.id) AS user_id,
    CASE 
        WHEN prev.id IS NULL THEN 'INSERTED'
        WHEN curr.id IS NULL THEN 'DELETED'
        WHEN curr.city != prev.city OR curr.name != prev.name THEN 'UPDATED'
        ELSE 'UNCHANGED'
    END AS change_type,
    prev.city AS old_city,
    curr.city AS new_city,
    prev.name AS old_name,
    curr.name AS new_name
FROM 
    (SELECT * FROM course.test.users VERSION AS OF <PASTE END_SNAPSHOT_ID>) AS curr
FULL OUTER JOIN 
    (SELECT * FROM course.test.users VERSION AS OF <PASTE START_SNAPSHOT_ID>) AS prev
ON curr.id = prev.id
WHERE 
    prev.id IS NULL OR curr.id IS NULL OR curr.city != prev.city OR curr.name != prev.name
""").show()

In [ ]:
spark.sql("""
SELECT * 
FROM course.test.users.changes
""").show()

In [ ]:
spark.sql("""
CALL course.system.create_changelog_view(
  table => 'test.users',
  identifier_columns => array('id'),
  compute_updates => false,
  options => map('start-snapshot-id','<PASTE SNAPSHOT_ID HERE>')
)
""").show(truncate=False)

In [ ]:
spark.sql("""
select * from users_changes
""").show()